# 03 — ARIMA / ARIMAX Baseline
Auto-select ARIMA order, add exogenous covariates (ARIMAX), rolling out-of-sample forecast.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

## 1. Load features

In [ ]:
train = pd.read_csv('../data/processed/train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv('../data/processed/val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv('../data/processed/test.csv',  index_col=0, parse_dates=True)

TARGET = 'silver_return'
EXOG   = ['gold_return', 'usd_return', 'copper_return', 'sp500_return']

y_train = train[TARGET].dropna()
y_test  = test[TARGET].dropna()
print(y_train.shape, y_test.shape)

## 2. Order selection via AIC grid search

In [ ]:
best_aic, best_order = np.inf, (1, 0, 1)
results = []

for p in range(0, 5):
    for q in range(0, 5):
        try:
            m = ARIMA(y_train, order=(p, 0, q)).fit()
            results.append({'p': p, 'q': q, 'aic': m.aic, 'bic': m.bic})
            if m.aic < best_aic:
                best_aic, best_order = m.aic, (p, 0, q)
        except Exception:
            pass

results_df = pd.DataFrame(results).sort_values('aic').head(10)
print(f'Best ARIMA order: {best_order}  AIC: {best_aic:.2f}')
results_df

## 3. Fit ARIMA on train, 1-step rolling forecast on test

In [ ]:
def rolling_forecast(y_train, y_test, order, exog_train=None, exog_test=None):
    """Expand-window 1-step-ahead forecast."""
    history = list(y_train)
    preds   = []
    exog_h  = list(exog_train.values) if exog_train is not None else None

    for t in range(len(y_test)):
        exog_f = None
        if exog_h is not None:
            exog_f = [exog_test.iloc[t].values]
        try:
            model  = ARIMA(history, order=order,
                           exog=exog_h if exog_h is not None else None).fit()
            fc     = model.forecast(steps=1, exog=exog_f)
            preds.append(float(fc.iloc[0]))
        except Exception:
            preds.append(np.nan)
        history.append(float(y_test.iloc[t]))
        if exog_h is not None:
            exog_h.append(exog_test.iloc[t].values)
    return np.array(preds)

print('Running ARIMA rolling forecast...')
preds_arima = rolling_forecast(y_train, y_test, best_order)
print('Done')

## 4. ARIMAX — add exogenous variables

In [ ]:
exog_cols = [c for c in EXOG if c in train.columns]
X_train = train[exog_cols].reindex(y_train.index).fillna(0)
X_test  = test[exog_cols].reindex(y_test.index).fillna(0)

print('Running ARIMAX rolling forecast...')
preds_arimax = rolling_forecast(y_train, y_test, best_order,
                                exog_train=X_train, exog_test=X_test)
print('Done')

## 5. Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

actuals = y_test.values
mask = ~np.isnan(preds_arima) & ~np.isnan(actuals)

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    # Directional accuracy
    da = np.mean(np.sign(y_true) == np.sign(y_pred))
    print(f'{name:15s}  RMSE={rmse:.6f}  MAE={mae:.6f}  Dir.Acc={da:.3f}')
    return {'model': name, 'rmse': rmse, 'mae': mae, 'dir_acc': da}

metrics = []
metrics.append(evaluate('Naive (t-1)',  actuals[mask][1:], actuals[mask][:-1]))
metrics.append(evaluate('ARIMA',        actuals[mask],     preds_arima[mask]))
metrics.append(evaluate('ARIMAX',       actuals[mask],     preds_arimax[mask]))

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv('../data/processed/metrics_arima.csv', index=False)
metrics_df

## 6. Forecast plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test.index, actuals, label='Actual', lw=1)
ax.plot(y_test.index, preds_arima,  label='ARIMA',  lw=1, alpha=0.8)
ax.plot(y_test.index, preds_arimax, label='ARIMAX', lw=1, alpha=0.8)
ax.set_title('1-step Rolling Forecast — Test Set')
ax.set_ylabel('Silver log-return')
ax.legend()
plt.tight_layout()
plt.show()